## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from statistics import mean
import warnings

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from scipy.signal import get_window
from scipy.fft import rfft, rfftfreq
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["test_noise_floor_26.04.24"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    signals_df = signals_df.astype("int32")
    signals_np = signals_df.to_numpy()
    # baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + 2 ** 14  # flip signals
    # signals_np = signals_np[:, :noise_sample_n]
    signals_df.columns = signals_df.columns.map(int)
    signals_df = pd.DataFrame(
        signals_np,
        index=signals_df.index,
        columns=signals_df.columns
    )
    exp_data["signals_df"] = signals_df

In [ ]:
noise_sample_n = 40
sample_period = 2e-9
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    # signals_df = signals_df.astype("int32")
    signals_np = signals_df.to_numpy()
    # # baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    # signals_np = -signals_np + 2 ** 14  # flip signals
    noise_np = signals_np[:, :noise_sample_n]
    # signals_df.columns = signals_df.columns.map(int)
    noise_df = pd.DataFrame(
        noise_np,
        index=signals_df.index,
        columns=signals_df.columns[:noise_sample_n]
    )
    exp_data["noise_df"] = noise_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["noise_df"]
    # print(signals_df.head())
    signals_np = signals_df.to_numpy()
    ffts = [rfft(signal) for signal in signals_np]
    ffts = [np.abs(fft) for fft in ffts]
    exp_data["noise_ffts"] = ffts
    # print(ffts[:5])
    ffts_pow = np.pow(ffts, 2) / 2
    exp_data["noise_pow"] = ffts_pow
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="divide by zero")
        fft_dbs = [10 * np.log10(fft) for fft in ffts_pow]
        fft_dbs = [
            np.nan_to_num(fft, nan=np.nan, posinf=np.nan, neginf=np.nan)
            for fft in fft_dbs
        ]
    # print(fft_dbs[:5])
    exp_data["noise_db"] = fft_dbs
#     signals_db_df = 10 * np.log10(signals_df)
#     print(signals_db_df.head())
#     exp_data["noise_df_db"] = signals_db_df

In [ ]:
blackman_window = get_window("blackman", noise_sample_n)
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["noise_df"]
    signals_np = signals_df.to_numpy()
    windowed_signals = [signal*blackman_window for signal in signals_np]
    signal_ffts = [rfft(signal) for signal in windowed_signals]
    signal_ffts = [2 / noise_sample_n * np.abs(signal) for signal in signal_ffts]
    squared_ffts = [np.pow(fft, 2) for fft in signal_ffts]
    all_ffts = np.stack(squared_ffts, axis=1)
    psd = np.mean(all_ffts, axis=1) * sample_period
    psd_db = 10 * np.log10(psd)
    exp_data["noise_psd"] = psd
    exp_data["noise_psd_db"] = psd_db

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
f = rfftfreq(noise_sample_n, sample_period)
f = f / 1e6

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    fig, ax = plt.subplots(figsize=(12, 6))
    print(signals_df.shape)
    # print(signals_df.head())
    x = np.arange(0, signals_df.shape[1]) * 2
    goal_percent = 5
    for i, signal in signals_df.iterrows():
        # if i == 0:
        #     print(signal)
        percent_done = (i / signals_df.shape[0]) * 100
        if percent_done >= goal_percent:
            print(f"{percent_done:.1f}")
            goal_percent += 5
        ax.plot(x, signal, lw=3, color="black", alpha=0.1)
    ax.set_xlim(50, 100)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Magnitude (ADC channel)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    print("Plot setup done, now drawing...")

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    ffts = exp_data["noise_ffts"]
    fig, ax = plt.subplots(figsize=(12, 6))
    for fft in ffts:
        ax.plot(f, fft, ".", color=bg_blue)
    ax.set_xlabel("Frequency (MHz)", fontsize=fontsize)
    ax.set_ylabel("Magnitude (ADC channels)", fontsize=fontsize)
    # ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1e9:.1f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    plt.grid()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    ffts = exp_data["noise_pow"]
    fig, ax = plt.subplots(figsize=(12, 6))
    for fft in ffts:
        # fft = np.pow(fft, 2) / 2
        ax.plot(f, fft, ".", color=bg_blue)
    ax.set_xlabel("Frequency (MHz)", fontsize=fontsize)
    ax.set_ylabel("Power (ADC channels^2)", fontsize=fontsize)
    # ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1e9:.1f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    plt.grid()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    ffts = exp_data["noise_db"]
    # print(ffts[:5])
    fig, ax = plt.subplots(figsize=(12, 6))
    for fft in ffts:
        ax.plot(f, fft, ".", color=bg_blue)
    ax.set_xlabel("Frequency (Hz)", fontsize=fontsize)
    ax.set_ylabel("Magnitude (dB)", fontsize=fontsize)
    # ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1e9:.1f}")
    ax.set_xscale("log")
    # ax.set_yscale("log")
    plt.grid()

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd = exp_data["noise_psd"]
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(f, psd, ".", color=bg_blue)
    ax.set_xlabel("Frequency (MHz)", fontsize=fontsize)
    ax.set_ylabel("PSD (ADC channel^2 Hz^-1)", fontsize=fontsize)
    ax.set_xscale("log")
    ax.set_yscale("log")
    plt.grid()

In [ ]:
# for exp_id, exp_data in experiment_neutron_data.items():
#     psd = exp_data["noise_psd_db"]
#     fig, ax = plt.subplots(figsize=(12, 6))
#     ax.plot(f, psd, ".", color=bg_blue)
#     ax.set_xlabel("Frequency (MHz)", fontsize=fontsize)
#     ax.set_ylabel("PSD (dB)", fontsize=fontsize)
#     ax.set_xscale("log")
#     plt.grid()